# Perturbation-budget (epsilon) sweep

Addresses the reviewer note that a fixed large budget (8/255) may be inappropriate for non-robust models (Shao et al., 2022). We sweep **eps in {1,2,4,8,16}/255**, craft **FGSM & PGD** on the same clean images, and report **ASR** and **HF-Energy detection AUROC** (learning-free, clean-calibrated, on successfully-attacked inputs only) — pristine and +hard-negative — for CIFAR-10 (upscaled x7) and ImageNet (native 224).

**Run:** in `/home/jupyter/SCAN`, Run All. Reuses the clean images from the `mixed_dataset.pkl` caches and the fine-tuned / ImageNet backbones. Output: `epsilon_sweep_results.json` + a summary table.

**Purpose:** show (1) detection scales with the budget (not an artifact of eps=8/255), and (2) the upscaled-vs-native gap and hard-negative collapse persist across budgets (budget-robust diagnosis).

In [ ]:
# ============ [PREAMBLE] self-contained helpers (verbatim from centerpiece harness) ============
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']; res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1).to(device); std=torch.tensor(s).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    cfg={'CIFAR-10':('resnet50_cifar10_finetuned.pt',10),'CIFAR-100':('resnet50_cifar100_finetuned.pt',100),
         'SVHN':('resnet50_svhn_finetuned.pt',10),'TinyImageNet':('resnet50_tinyimagenet_finetuned.pt',200)}
    if ds in cfg:
        ck=(_find(cfg[ds][0]) or [None])[0]; m=models.resnet50(weights=None); m.fc=nn.Linear(2048,cfg[ds][1])
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict'])
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75):
    return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet',p)
    return out
def auc_ci(neg,pos,B=2000,seed=SEED):
    neg=np.asarray(neg);pos=np.asarray(pos)
    if len(pos)<3 or len(neg)<3: return float('nan'),float('nan'),float('nan')
    base=roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))],np.r_[neg,pos])
    rng=np.random.RandomState(seed); b=[]
    for _ in range(B):
        nb=neg[rng.randint(0,len(neg),len(neg))]; pb=pos[rng.randint(0,len(pos),len(pos))]
        b.append(roc_auc_score(np.r_[np.zeros(len(nb)),np.ones(len(pb))],np.r_[nb,pb]))
    return float(base),float(np.percentile(b,2.5)),float(np.percentile(b,97.5))
def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]

# --- core detector (paper's HF-Energy, learning-free) ---
def feat_hfe(b):
    return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()

# --- attacks: FGSM (verbatim) + standard L-inf PGD ---
def fgsm(x, y, bb, pp, eps255=8.0):
    x=x.clone().to(device).requires_grad_(True)
    loss=F.cross_entropy(bb(pp(x)),y.to(device)); g,=torch.autograd.grad(loss,x)
    return (x+eps255*g.sign()).clamp(0,255).detach()
def pgd(x, y, bb, pp, eps255=8.0, steps=20, alpha255=None):
    a=alpha255 if alpha255 is not None else max(1.0, eps255/4.0)
    x=x.to(device); y=y.to(device)
    xa=(x+torch.empty_like(x).uniform_(-eps255,eps255)).clamp(0,255).detach()
    for _ in range(steps):
        xa.requires_grad_(True)
        loss=F.cross_entropy(bb(pp(xa)),y); g,=torch.autograd.grad(loss,xa)
        xa=(xa+a*g.sign()).clamp(x-eps255,x+eps255).clamp(0,255).detach()
    return xa
print('[PREAMBLE] helpers + attacks ready; device =', device)


In [ ]:
# ============ [EPSILON SWEEP] does detection / the upscaling artifact depend on the budget? ============
# For each epsilon we craft FGSM & PGD on the SAME clean images, record ASR, and measure HF-Energy
# detection AUROC (learning-free, clean-calibrated) on SUCCESSFULLY-attacked inputs only (paper Sec. 4),
# pristine and +hard-negative, for CIFAR-10 (upscaled x7) and ImageNet (native 224).
EPS255_LIST=[1.0,2.0,4.0,8.0,16.0]; N_IMG=300; FPRS=[0.01,0.05,0.10]
MX=find_mixed(); assert MX, 'no mixed_dataset.pkl found'
print('using caches:', MX)

def hfe_scores(X, mu, sd, bs=64):
    out=[]
    for i in range(0,len(X),bs): out.append(feat_hfe(X[i:i+bs].to(device)))
    v=np.concatenate(out); return np.abs((v-mu)/sd)

EPS_RESULTS={}
for ds,pkl in MX.items():
    regime='upscaled x7' if ds=='CIFAR-10' else 'native 224'
    bb=load_backbone(ds); pp=make_pp(ds)
    mixed=pickle.load(open(pkl,'rb'))
    clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
    rng=np.random.RandomState(SEED)
    clean=[clean[i] for i in rng.permutation(len(clean))[:N_IMG]]
    Xc=torch.cat(clean,0)
    with torch.no_grad(): y=bb(pp(Xc.to(device))).argmax(1)          # model's clean prediction (untargeted ref)
    # clean calibration for HF-Energy |z|
    ci,ti=half(len(clean)); Xcal=Xc[ci]; Xcte=Xc[ti]
    fcal=feat_hfe(Xcal.to(device)); mu=fcal.mean(); sd=fcal.std()+1e-8
    s_clean_te=hfe_scores(Xcte,mu,sd)
    # hard negatives from clean test half (noise8 + JPEG75 + blur1)
    hn=torch.cat([(Xcte+torch.randn_like(Xcte)*8.0).clamp(0,255),
                  jpeg_batch(Xcte.to(device)).cpu(),
                  gb(Xcte.to(device),1.0).clamp(0,255).cpu()],0)
    s_hn=hfe_scores(hn,mu,sd)

    EPS_RESULTS[ds]={'regime':regime,'sweep':{}}
    for eps in EPS255_LIST:
        row={}
        for atk_name,atk in [('FGSM',fgsm),('PGD',pgd)]:
            advs=[]
            for i in range(0,len(Xc),64):
                xb=Xc[i:i+64].to(device); yb=y[i:i+64]
                advs.append(atk(xb,yb,bb,pp,eps255=eps).cpu())
            Xadv=torch.cat(advs,0)
            with torch.no_grad(): yhat=bb(pp(Xadv.to(device))).argmax(1)
            succ=(yhat!=y).cpu().numpy()                              # successfully attacked
            asr=float(succ.mean())
            s_adv=hfe_scores(Xadv,mu,sd)[succ]                        # detection on successful only
            a_p,_,_=auc_ci(s_clean_te,s_adv)
            a_h,_,_=auc_ci(np.r_[s_clean_te,s_hn],s_adv)
            row[atk_name]={'asr':round(asr,3),'n_succ':int(succ.sum()),
                           'pristine_auroc':None if np.isnan(a_p) else round(a_p,4),
                           'hardneg_auroc':None if np.isnan(a_h) else round(a_h,4)}
        EPS_RESULTS[ds]['sweep'][f'{eps/255:.4f}']={'eps255':eps, **row}
        f=row['FGSM']; p=row['PGD']
        print(f'[{ds:8s} {regime:11s}] eps={eps:>4.0f}/255  '
              f'FGSM asr={f["asr"]:.2f} HF={f["pristine_auroc"]} (+hn {f["hardneg_auroc"]})  |  '
              f'PGD asr={p["asr"]:.2f} HF={p["pristine_auroc"]} (+hn {p["hardneg_auroc"]})')

json.dump(EPS_RESULTS, open('epsilon_sweep_results.json','w'), indent=2)
print('\nsaved epsilon_sweep_results.json')


In [ ]:
# ============ [SUMMARY] epsilon-sweep table + interpretation ============
import pandas as pd
rows=[]
for ds in EPS_RESULTS:
    reg=EPS_RESULTS[ds]['regime']
    for k,v in EPS_RESULTS[ds]['sweep'].items():
        rows.append({'Dataset':f'{ds} ({reg})','eps':f'{v["eps255"]:.0f}/255',
                     'FGSM ASR':v['FGSM']['asr'],'FGSM HF-AUROC':v['FGSM']['pristine_auroc'],'FGSM +hardneg':v['FGSM']['hardneg_auroc'],
                     'PGD ASR':v['PGD']['asr'],'PGD HF-AUROC':v['PGD']['pristine_auroc'],'PGD +hardneg':v['PGD']['hardneg_auroc']})
df=pd.DataFrame(rows)
print(df.to_string(index=False))
print('\n% ---- paste-ready LaTeX rows: eps & FGSM_ASR & FGSM_HF & PGD_ASR & PGD_HF (per dataset) ----')
for _,r in df.iterrows():
    print(f"  {r['eps']:>7s} & {r['FGSM ASR']:.2f} & {r['FGSM HF-AUROC']} & {r['PGD ASR']:.2f} & {r['PGD HF-AUROC']} \\\\  % {r['Dataset']}")
print('\nWHAT TO LOOK FOR:')
print(' (1) budget dependence: as eps decreases, ASR and HF-Energy detection AUROC decrease together')
print('     (smaller perturbation -> weaker high-frequency signal) -> our detection is not an artifact of large eps.')
print(' (2) artifact persists across eps: at every budget, upscaled CIFAR stays much higher than native ImageNet,')
print('     and +hard-negative collapses the score -> the upscaling-artifact diagnosis is budget-robust.')
